In [1]:
from pathlib import Path
import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

CONFIG = {
    "output_dir": Path("/content/drive/MyDrive/doculens_lava/outputs"),
    "pages_path": Path("/content/drive/MyDrive/doculens_lava/outputs/pages.parquet"),
    "train_path": Path("/content/drive/MyDrive/doculens_lava/outputs/train.csv"),
    "test_path": Path("/content/drive/MyDrive/doculens_lava/outputs/test.csv"),
    "embeddings_path": Path("/content/drive/MyDrive/doculens_lava/outputs/page_embeddings.npy"),
    "embed_model": "BAAI/bge-m3",
    "top_k": 5,
}

if not torch.cuda.is_available():
    raise RuntimeError("GPU is OFF. Go to Runtime -> Change runtime type -> T4 GPU.")

pages_df = pd.read_parquet(CONFIG["pages_path"])
train_df = pd.read_csv(CONFIG["train_path"])
test_df = pd.read_csv(CONFIG["test_path"])

print("GPU active:", torch.cuda.get_device_name(0))
print("Pages loaded:", len(pages_df), "| Empty text pages:", int((pages_df["text"].str.len() == 0).sum()))
print("Train questions:", len(train_df), "| Test questions:", len(test_df))

Mounted at /content/drive
GPU active: Tesla T4
Pages loaded: 4772 | Empty text pages: 46
Train questions: 16 | Test questions: 624


In [2]:
!pip install -q sentence-transformers

import time
import numpy as np
from sentence_transformers import SentenceTransformer

# Load from Drive if already computed so we never waste GPU time twice.
if CONFIG["embeddings_path"].exists():
    page_embeddings = np.load(CONFIG["embeddings_path"])
    print("Loaded saved embeddings from Drive:", page_embeddings.shape)
else:
    start_time = time.time()
    model = SentenceTransformer(CONFIG["embed_model"], device="cuda")
    model.max_seq_length = 1024  # Prevents T4 GPU out-of-memory on dense pages.
    model.half()                 # FP16 doubles speed on T4 with no retrieval loss.

    texts = pages_df["text"].fillna("").astype(str).tolist()
    page_embeddings = model.encode(
        texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True,  # Normalized vectors let us use dot product as cosine similarity.
    )

    page_embeddings = np.asarray(page_embeddings, dtype=np.float32)
    np.save(CONFIG["embeddings_path"], page_embeddings)
    minutes = (time.time() - start_time) / 60
    print(f"Saved embeddings to Drive: {page_embeddings.shape} ({minutes:.2f} min)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Saved embeddings to Drive: (4772, 1024) (3.04 min)


In [3]:
import ast
import numpy as np
from sentence_transformers import SentenceTransformer

# Reuse the model already in GPU memory, or load it if you restarted the runtime.
if "model" not in globals():
    model = SentenceTransformer(CONFIG["embed_model"], device="cuda")
    model.max_seq_length = 1024
    model.half()

# Pick the question column safely in case the CSV header is 'question' or 'query'.
q_col = "question" if "question" in train_df.columns else "query"
train_questions = train_df[q_col].fillna("").astype(str).tolist()

q_embeddings = model.encode(
    train_questions,
    batch_size=16,
    normalize_embeddings=True,
)
q_embeddings = np.asarray(q_embeddings, dtype=np.float32)

# Track Recall@K and Full-Hit@K (all evidence pages found) for K = 1, 3, 5.
k_values = [1, 3, 5]
recalls = {k: [] for k in k_values}
full_hits = {k: [] for k in k_values}
misses = []

for i in range(len(train_df)):
    row = train_df.iloc[i]
    file_id = row["file_id"]
    true_pages = ast.literal_eval(str(row["evidence_page_number"]))

    # Restrict search to pages belonging to this exact PDF.
    mask = (pages_df["file_id"] == file_id) & (pages_df["split"] == "train")
    doc_indices = pages_df.index[mask].to_numpy()
    doc_pages = pages_df.loc[doc_indices, "page_number"].to_numpy()

    # Cosine similarity = dot product because both vectors are normalized.
    scores = page_embeddings[doc_indices] @ q_embeddings[i]
    order = np.argsort(scores)[::-1]
    ranked_pages = [int(doc_pages[idx]) for idx in order]

    for k in k_values:
        top_k = set(ranked_pages[:k])
        found = len(top_k.intersection(true_pages))
        recalls[k].append(found / max(1, len(true_pages)))
        full_hits[k].append(1.0 if found == len(true_pages) else 0.0)

    if not set(true_pages).issubset(set(ranked_pages[:5])):
        misses.append((row["id"], file_id, true_pages, ranked_pages[:5]))

print("=== DENSE RETRIEVAL BASELINE (BGE-M3) ON TRAIN (16 QUESTIONS) ===")
for k in k_values:
    print(f"Top-{k} | Page Recall: {np.mean(recalls[k]) * 100:.1f}% | All-Pages Hit: {np.mean(full_hits[k]) * 100:.1f}%")

print(f"\nMissed at least 1 page in Top-5: {len(misses)} / {len(train_df)}")
for item in misses:
    print(" - id:", item[0], "| file:", item[1], "| true:", item[2], "| pred_top5:", item[3])

=== DENSE RETRIEVAL BASELINE (BGE-M3) ON TRAIN (16 QUESTIONS) ===
Top-1 | Page Recall: 56.8% | All-Pages Hit: 37.5%
Top-3 | Page Recall: 79.2% | All-Pages Hit: 62.5%
Top-5 | Page Recall: 89.1% | All-Pages Hit: 81.2%

Missed at least 1 page in Top-5: 3 / 16
 - id: q_0003 | file: j_0158 | true: [13, 21] | pred_top5: [1, 20, 13, 16, 4]
 - id: q_0013 | file: j_0060 | true: [6] | pred_top5: [13, 3, 1, 2, 4]
 - id: q_0015 | file: v_0104 | true: [1, 2, 3, 15] | pred_top5: [3, 16, 1, 2, 8]


In [4]:
import ast
import numpy as np
from sentence_transformers import CrossEncoder

# Load the multilingual reranker in FP16 so it runs fast on T4 GPU.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=1024)
reranker.model.half()

k_values = [1, 3, 5]
recalls = {k: [] for k in k_values}
full_hits = {k: [] for k in k_values}
misses = []

for i in range(len(train_df)):
    row = train_df.iloc[i]
    file_id = row["file_id"]
    question = str(row[q_col])
    true_pages = ast.literal_eval(str(row["evidence_page_number"]))

    # Step 1: Get Top-10 candidate pages from BGE-M3 inside this exact PDF.
    mask = (pages_df["file_id"] == file_id) & (pages_df["split"] == "train")
    doc_indices = pages_df.index[mask].to_numpy()
    scores = page_embeddings[doc_indices] @ q_embeddings[i]
    top10_indices = doc_indices[np.argsort(scores)[::-1][:10]]

    # Step 2: Re-score each (question, page_text) pair with the cross-encoder.
    pairs = []
    candidate_pages = []
    for idx in top10_indices:
        page_text = str(pages_df.loc[idx, "text"])
        pairs.append([question, page_text])
        candidate_pages.append(int(pages_df.loc[idx, "page_number"]))

    rerank_scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)
    order = np.argsort(rerank_scores)[::-1]
    ranked_pages = [candidate_pages[pos] for pos in order]

    # Step 3: Measure Recall@K and All-Pages Hit@K.
    for k in k_values:
        top_k = set(ranked_pages[:k])
        found = len(top_k.intersection(true_pages))
        recalls[k].append(found / max(1, len(true_pages)))
        full_hits[k].append(1.0 if found == len(true_pages) else 0.0)

    if not set(true_pages).issubset(set(ranked_pages[:5])):
        misses.append((row["id"], file_id, true_pages, ranked_pages[:5]))

print("=== RERANKED RETRIEVAL (BGE-M3 + RERANKER-V2-M3) ON TRAIN ===")
for k in k_values:
    print(f"Top-{k} | Page Recall: {np.mean(recalls[k]) * 100:.1f}% | All-Pages Hit: {np.mean(full_hits[k]) * 100:.1f}%")

print(f"\nMissed at least 1 page in Top-5: {len(misses)} / {len(train_df)}")
for item in misses:
    print(" - id:", item[0], "| file:", item[1], "| true:", item[2], "| pred_top5:", item[3])

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

=== RERANKED RETRIEVAL (BGE-M3 + RERANKER-V2-M3) ON TRAIN ===
Top-1 | Page Recall: 50.0% | All-Pages Hit: 31.2%
Top-3 | Page Recall: 88.5% | All-Pages Hit: 81.2%
Top-5 | Page Recall: 98.4% | All-Pages Hit: 93.8%

Missed at least 1 page in Top-5: 1 / 16
 - id: q_0015 | file: v_0104 | true: [1, 2, 3, 15] | pred_top5: [8, 3, 2, 15, 16]


In [5]:
import time
import numpy as np


def retrieve_top5_for_split(df, split_name):
    """
    Input: Question DataFrame (train_df or test_df) and split label ('train' or 'test').
    Output: Copy of df with two new columns: 'top5_pages' and 'top5_scores'.
    """
    q_col = "question" if "question" in df.columns else "query"
    questions = df[q_col].fillna("").astype(str).tolist()

    # Encode all questions at once on GPU for speed.
    q_embeddings = model.encode(
        questions,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    q_embeddings = np.asarray(q_embeddings, dtype=np.float32)

    all_top5_pages = []
    all_top5_scores = []
    start_time = time.time()

    for i in range(len(df)):
        file_id = str(df.iloc[i]["file_id"]).replace(".pdf", "")
        mask = (pages_df["file_id"] == file_id) & (pages_df["split"] == split_name)
        doc_indices = pages_df.index[mask].to_numpy()

        if len(doc_indices) == 0:
            raise ValueError(f"No pages found for {split_name} file_id: {file_id}")

        # Stage 1: Fast dense retrieval (Top-10 pages inside this PDF).
        dense_scores = np.atleast_1d(page_embeddings[doc_indices] @ q_embeddings[i])
        top10_indices = doc_indices[np.argsort(dense_scores)[::-1][:10]]

        # Stage 2: Cross-encoder reranking on those Top-10 pages.
        pairs = []
        candidate_pages = []
        for idx in top10_indices:
            pairs.append([questions[i], str(pages_df.loc[idx, "text"])])
            candidate_pages.append(int(pages_df.loc[idx, "page_number"]))

        # atleast_1d prevents a crash when a PDF has only 1 page.
        rerank_scores = np.atleast_1d(
            reranker.predict(pairs, batch_size=16, show_progress_bar=False)
        )
        order = np.argsort(rerank_scores)[::-1][:5]

        all_top5_pages.append([candidate_pages[pos] for pos in order])
        all_top5_scores.append([float(rerank_scores[pos]) for pos in order])

        if (i + 1) % 100 == 0 or (i + 1) == len(df):
            elapsed = time.time() - start_time
            print(f"[{split_name}] Processed {i + 1}/{len(df)} questions ({elapsed:.1f}s)")

    result_df = df.copy()
    result_df["top5_pages"] = all_top5_pages
    result_df["top5_scores"] = all_top5_scores
    return result_df


train_retrieved = retrieve_top5_for_split(train_df, "train")
test_retrieved = retrieve_top5_for_split(test_df, "test")

train_save_path = CONFIG["output_dir"] / "retrieved_train.parquet"
test_save_path = CONFIG["output_dir"] / "retrieved_test.parquet"

train_retrieved.to_parquet(train_save_path, index=False)
test_retrieved.to_parquet(test_save_path, index=False)

print("\n=== RETRIEVAL COMPLETE ===")
print("Saved train:", train_save_path, "| rows:", len(train_retrieved))
print("Saved test:", test_save_path, "| rows:", len(test_retrieved))
print("\nFirst 3 test predictions:")
print(test_retrieved[["id", "file_id", "top5_pages"]].head(3))

[train] Processed 16/16 questions (5.8s)
[test] Processed 100/624 questions (37.6s)
[test] Processed 200/624 questions (71.9s)
[test] Processed 300/624 questions (105.8s)
[test] Processed 400/624 questions (139.7s)
[test] Processed 500/624 questions (175.9s)
[test] Processed 600/624 questions (210.2s)
[test] Processed 624/624 questions (216.6s)

=== RETRIEVAL COMPLETE ===
Saved train: /content/drive/MyDrive/doculens_lava/outputs/retrieved_train.parquet | rows: 16
Saved test: /content/drive/MyDrive/doculens_lava/outputs/retrieved_test.parquet | rows: 624

First 3 test predictions:
       id file_id            top5_pages
0  q_0016  j_0027  [14, 15, 16, 17, 10]
1  q_0017  j_0162   [12, 11, 8, 29, 10]
2  q_0018  j_0011       [3, 4, 5, 6, 7]
